# Iterators

Every type in this chapter has been an ADT you interact with through explicit operations: `Push`, `Pop`, `Get`, `Set`. `foreach` is different — it is a loop that lets a *type* decide what "give me the next element" means, without the caller ever touching an index or a key. This section covers how that contract works, and how to implement it for your own types.

```{contents}
:local:
:depth: 2
```

```{index} IEnumerable; IEnumerator
```



```{index} Why foreach Works
```

## Why foreach Works




`foreach` is not built-in magic tied to `List<T>` or `Dictionary<TKey, TValue>` specifically. It works on *any* type that implements `IEnumerable<T>` — the interface that promises "I can hand out an object that walks through my elements one at a time." That helper object implements `IEnumerator<T>`, with a `MoveNext()` method and a `Current` property.

```csharp
interface IEnumerable<T>
{
    IEnumerator<T> GetEnumerator();
}

interface IEnumerator<T>
{
    T Current { get; }
    bool MoveNext();
}
```

Written out, `foreach (var item in collection)` is shorthand for a loop built from exactly these two members:

```csharp
var e = collection.GetEnumerator();
while (e.MoveNext())
{
    var item = e.Current;
    // loop body
}
```

This is the ADT pattern from earlier sections applied to iteration itself: `IEnumerable<T>` is a contract, and any representation — an array, a linked structure, a database cursor — can satisfy it.

For related explanation, see the further reading below.[^development-reading]


```{index} Separate Producing a Sequence from Storing One
```



```{index} Separate Producing a Sequence from Storing One
```

## Separate Producing a Sequence from Storing One



A collection knows where its elements are stored. A client often needs only to visit those elements in the promised order. Enumeration provides a boundary between those responsibilities. Foreach asks for an enumeration protocol rather than directly reading private array slots or node links.

For a simple countable sequence 2, 4, 6, imagine an enumerator positioned before the first value. A successful MoveNext advances to the next value and permits reading Current. After the final value, MoveNext reports false. The client does not invent an extra zero value merely because iteration has ended. This success/result separation resembles the Try operations from earlier collection lessons.

| Request | Outcome | Current value to consume |
| --- | --- | --- |
| First MoveNext | true | 2 |
| Second MoveNext | true | 4 |
| Third MoveNext | true | 6 |
| Fourth MoveNext | false | No next element |

An IEnumerable<T> describes something that can provide enumeration. An IEnumerator<T> represents progress through one enumeration. They are not interchangeable names for a collection. Multiple enumeration requests can have separate progress states, and a producer may compute elements instead of keeping all of them in a list.




```{index} A Yield Point Suspends Rather than Finishes the Sequence
```



```{index} A Yield Point Suspends Rather than Finishes the Sequence
```

## A Yield Point Suspends Rather than Finishes the Sequence



The pattern `IEnumerable<T> Produce(...) { ... yield return value; ... }` defines an iterator. Yielding supplies one element and suspends that enumeration's progress until the consumer requests another. An ordinary return from a value-returning method finishes that call; yield return participates in a sequence of resumptions. Explain this distinction before treating yield as abbreviated list construction.

Laziness changes when work and failures occur. Obtaining an enumerable need not run every statement that will produce its values. If the consumer stops after two elements, later elements may never be computed. An expensive calculation inside an iterator is not paid simply because an enumerable variable has been declared, but repeated enumerations may repeat that work. Do not promise caching unless the implementation explicitly provides it.

The producer also needs a mutation policy. An enumeration over shared mutable storage can be affected by changes made after the enumerable is obtained. Library collections may detect invalid mutation during enumeration; custom types must state their own behavior. Returning IEnumerable<T> does not automatically create a snapshot or make private storage immutable.

Guided prediction: an iterator announces each produced number and yields 2, 4, 6. A caller breaks after consuming 4. Which production announcements should appear? Then run a second enumeration and ask whether it starts with a fresh progress state. Use the existing runnable iterator examples to check your prediction and distinguish fresh enumeration from continuing one enumerator.


```{index} yield return
```



```{index} Writing an Iterator with yield return
```

## Writing an Iterator with yield return




Implementing `IEnumerator<T>` by hand — tracking a position field, writing `MoveNext()` and `Current` yourself — is tedious for something as simple as "count from one number to another." The `yield return` statement lets the compiler generate that machinery for you. A method that contains `yield return` becomes an iterator: each call to `yield return` produces one element and pauses; the method resumes from that point the next time an element is requested.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
static IEnumerable<int> EvenNumbers(int first, int last)
{
    for (int n = first; n <= last; n++)
    {
        if (n % 2 == 0)
        {
            yield return n;
        }
    }
}

foreach (int n in EvenNumbers(1, 10))
{
    Console.Write(n + " ");
}

2 4 6 8 10 

Nothing about `EvenNumbers` mentions `IEnumerator`, `MoveNext`, or `Current` — the compiler builds a hidden class that implements all of it. The method's return type is `IEnumerable<int>`, and the body describes *what to produce*, not *how to track position*.

```{index} IEnumerable; custom implementation
```



```{index} Making Your Own Type foreach-able
```

## Making Your Own Type foreach-able




A standalone iterator *method* is useful, but sometimes the thing you want to iterate is a type you have already defined — an ADT you built, not a built-in collection. To make `foreach` work directly on an instance of your type, implement `IEnumerable<T>` and put a `yield return` inside `GetEnumerator()`:

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
using System.Collections;

var range = new NumberRange(5, 4);
foreach (int n in range)
{
    Console.Write(n + " ");
}

class NumberRange : IEnumerable<int>
{
    private readonly int start;
    private readonly int count;

    public NumberRange(int start, int count)
    {
        this.start = start;
        this.count = count;
    }

    public IEnumerator<int> GetEnumerator()
    {
        for (int i = 0; i < count; i++)
        {
            yield return start + i;
        }
    }

    IEnumerator IEnumerable.GetEnumerator() => GetEnumerator();
}

5 6 7 8 

`IEnumerable<T>` requires the generic `GetEnumerator()`. The second, non-generic `IEnumerator IEnumerable.GetEnumerator()` line is boilerplate C# requires because `IEnumerable<T>` itself extends the older, non-generic `IEnumerable` — it just forwards to the generic version above it. Once both are present, `NumberRange` works with `foreach`, LINQ, and any other API that accepts `IEnumerable<T>`, exactly like `List<T>` does.

```{index} Writing the Enumerator by Hand
```

### Writing the Enumerator by Hand

`yield return` hides a small object that most students benefit from seeing once. An enumerator is a *position object*: it remembers where one walk through the collection currently stands, separately from the collection itself. That separation is what lets two loops walk the same collection at the same time, or lets a walk pause and resume. Writing one by hand makes the contract explicit:

```csharp
class MyEnumerator : IEnumerator<T>
{
    private int position = -1;        // before the first element
    public T Current => ...;          // the element at position
    public bool MoveNext() { ... }    // advance; return false when past the end
    public void Reset() { ... }       // rarely used; may throw NotSupportedException
    public void Dispose() { }         // release resources, if any
}
```

The example implements `NumberRange` from above without `yield`. Compare the two versions line by line: everything the hand-written class tracks in fields, the compiler tracked for you in the earlier version.

In [ ]:
using System;
using System.Collections;
using System.Collections.Generic;

var range = new NumberRange(5, 4);
foreach (int n in range) Console.Write(n + " ");
Console.WriteLine();

// Two independent walks over the same range: each enumerator keeps its own position.
IEnumerator<int> first = range.GetEnumerator();
IEnumerator<int> second = range.GetEnumerator();
first.MoveNext(); first.MoveNext();
second.MoveNext();
Console.WriteLine($"first is at {first.Current}, second is at {second.Current}");

class NumberRange : IEnumerable<int>
{
    private readonly int start, count;
    public NumberRange(int start, int count) { this.start = start; this.count = count; }
    public IEnumerator<int> GetEnumerator() => new RangeEnumerator(start, count);
    IEnumerator IEnumerable.GetEnumerator() => GetEnumerator();

    private sealed class RangeEnumerator : IEnumerator<int>
    {
        private readonly int start, count;
        private int position = -1;                     // before the first element
        public RangeEnumerator(int start, int count) { this.start = start; this.count = count; }
        public int Current => start + position;
        object IEnumerator.Current => Current;
        public bool MoveNext()
        {
            if (position + 1 >= count) return false;   // nothing left to visit
            position++;
            return true;
        }
        public void Reset() => position = -1;
        public void Dispose() { }
    }
}

5 6 7 8 
first is at 6, second is at 5


The first line is `5 6 7 8`, identical to the `yield` version. The second line, `first is at 6, second is at 5`, is the property that matters: each call to `GetEnumerator` created a fresh `RangeEnumerator` with its own `position`, so advancing one did not move the other. The position starts at -1 so that the first `MoveNext` lands on element 0, which is why reading `Current` before any `MoveNext` is a contract violation; the hand-written class would return `start - 1`, a value that was never in the range.

| Concern | Hand-written enumerator | `yield return` iterator |
| --- | --- | --- |
| Where the position lives | An explicit field you declare | A hidden field the compiler generates |
| Advancing | Your `MoveNext` body | Resuming after the last `yield return` |
| End of sequence | Your `MoveNext` returns `false` | The method body finishes or hits `yield break` |
| Best used when | The walk needs unusual control, such as reset or random access | Almost always; it is shorter and harder to get wrong |

```{index} iterator; lazy evaluation
```



```{index} Iterators Are Lazy
```

## Iterators Are Lazy



An iterator method does not run its body when you call it — it runs one step at a time, only as far as the caller actually asks for elements. Calling `EvenNumbers(1, 1000000)` returns instantly; the loop body only executes as `foreach` requests each next value. This matters for two reasons: an iterator can describe an infinite or unbounded sequence without ever running forever, and a `foreach` that stops early (with `break`) never computes the elements it never asked for.

The example below makes the timing visible. The producer announces each value as it is about to yield it, and the consumer stops after the second value. Predict the order of the printed lines before running: in particular, decide whether `producing 6` ever appears, and whether the second `foreach` begins with `producing 2` again.

In [ ]:
using System;
using System.Collections.Generic;

static IEnumerable<int> Evens()
{
    foreach (int n in new[] { 2, 4, 6 })
    {
        Console.WriteLine($"  producing {n}");
        yield return n;
    }
    Console.WriteLine("  producer finished");
}

IEnumerable<int> sequence = Evens();          // nothing printed yet: no element was requested
Console.WriteLine("enumerable obtained");

foreach (int n in sequence)
{
    Console.WriteLine($"consumed {n}");
    if (n == 4) break;                        // stop early
}

Console.WriteLine("second enumeration");
foreach (int n in sequence) Console.WriteLine($"consumed {n}");

enumerable obtained
  producing 2
consumed 2
  producing 4
consumed 4
second enumeration
  producing 2
consumed 2
  producing 4
consumed 4
  producing 6
consumed 6
  producer finished


The first printed line is `enumerable obtained`: calling `Evens()` ran none of its body. Each `producing` line appears immediately before the matching `consumed` line, because the body runs only as far as the next `yield return` each time the loop asks for a value. After the consumer breaks at 4, `producing 6` and `producer finished` never appear in the first walk; the work for the third element was never paid for. The second `foreach` starts from `producing 2` again and this time runs to `producer finished`, because each `foreach` requests a fresh enumerator from the same enumerable. Two conclusions follow. An iterator that does expensive work per element costs nothing until that element is requested, and nothing at all for elements after an early exit. And an enumerable is a recipe, not a cache: enumerating it twice performs the work twice unless the implementation stores results deliberately.

```{index} Code Tracing: Predict the Interleaving
```

### Code Tracing: Predict the Interleaving

Trace the program below without running it and write the printed lines in order. The producer yields the running total of a list of daily sales, and the consumer stops as soon as the total reaches the target.

```csharp
static IEnumerable<int> RunningTotal(int[] sales)
{
    int total = 0;
    foreach (int s in sales)
    {
        total += s;
        Console.WriteLine($"  total so far {total}");
        yield return total;
    }
}

int[] sales = { 300, 450, 500, 250 };
foreach (int t in RunningTotal(sales))
{
    Console.WriteLine($"checked {t}");
    if (t >= 1000) { Console.WriteLine("target reached"); break; }
}
```

In [ ]:
// Solution
using System;
using System.Collections.Generic;

static IEnumerable<int> RunningTotal(int[] sales)
{
    int total = 0;
    foreach (int s in sales)
    {
        total += s;
        Console.WriteLine($"  total so far {total}");
        yield return total;
    }
}

int[] sales = { 300, 450, 500, 250 };
foreach (int t in RunningTotal(sales))
{
    Console.WriteLine($"checked {t}");
    if (t >= 1000) { Console.WriteLine("target reached"); break; }
}

  total so far 300
checked 300
  total so far 750
checked 750
  total so far 1250
checked 1250
target reached


Seven lines print, alternating producer and consumer: `total so far 300`, `checked 300`, `total so far 750`, `checked 750`, `total so far 1250`, `checked 1250`, `target reached`. The fourth sale, 250, is never added, so no `total so far 1500` line exists. A trace that prints all four producer lines first and then the consumer lines has treated the iterator as a list built in advance; the interleaving is the evidence that it is not.

```{index} Code Completion: Count Down
```

### Code Completion: Count Down

Complete `CountDown` so that it yields `from, from - 1, ..., 1` and then stops. The expected output is `5 4 3 2 1` on one line, and `CountDown(0)` must yield nothing. Remember that an iterator ends when its body finishes, so there is no value to `return`.

In [ ]:
using System;
using System.Collections.Generic;

foreach (int n in CountDown(5)) Console.Write(n + " ");
Console.WriteLine();
Console.WriteLine($"CountDown(0) yielded {Count(CountDown(0))} values");

static IEnumerable<int> CountDown(int from)
{
    // Your code starts here.

    // Your code ends here.
}

static int Count(IEnumerable<int> items)
{
    int n = 0;
    foreach (int _ in items) n++;
    return n;
}

In [ ]:
// Solution
using System;
using System.Collections.Generic;

foreach (int n in CountDown(5)) Console.Write(n + " ");
Console.WriteLine();
Console.WriteLine($"CountDown(0) yielded {Count(CountDown(0))} values");

static IEnumerable<int> CountDown(int from)
{
    for (int n = from; n >= 1; n--)
    {
        yield return n;
    }
}

static int Count(IEnumerable<int> items)
{
    int n = 0;
    foreach (int _ in items) n++;
    return n;
}

5 4 3 2 1 
CountDown(0) yielded 0 values


The loop condition `n >= 1` is the whole termination argument: when `from` is 0 the condition is false on the first test, the body never yields, and the sequence is empty. Writing `while (true)` with a `yield return n--` and no exit, or testing `n > 0` after decrementing, produces an endless or off-by-one sequence; `foreach` would never finish or would include 0. If the starter cell were left empty, the method would compile and yield nothing at all, because an iterator body with no `yield` is a legal empty sequence.

```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. `IEnumerable<Order> late = FindLate(orders);` runs instantly even for a million orders. Does that mean `FindLate` is fast?

```{dropdown} Answer
Not necessarily. If `FindLate` is an iterator, the call only created the enumerator; the scanning happens later, one element at a time, inside whatever loop consumes `late`. The cost moved; it did not disappear.
```

2. A method enumerates the same `IEnumerable<T>` twice and gets different results. Name two possible causes.

```{dropdown} Answer
The iterator may read a changing source, such as a file or a collection that another part of the program modified between the walks, or the iterator may itself have side effects, such as consuming input or advancing a counter. An enumerable promises a way to walk, not a stored snapshot.
```

3. Why does a type need both `IEnumerable<T>.GetEnumerator()` and the non-generic `IEnumerable.GetEnumerator()`?

```{dropdown} Answer
`IEnumerable<T>` extends the older non-generic `IEnumerable`, so a class implementing the generic interface must also satisfy the non-generic member. The usual implementation forwards it to the generic one; it exists for code written before generics.
```

4. What does `MoveNext` return the first time it is called on a fresh enumerator over an empty collection, and what must a consumer therefore never do?

```{dropdown} Answer
It returns `false`. The consumer must not read `Current` before a `MoveNext` that returned `true`; a position that was never advanced has no element, and what `Current` returns in that state is unspecified.
```

```{index} Practice
```

## Practice




1. Write an iterator method `static IEnumerable<int> CountDown(int from)` that yields `from, from - 1, ..., 1`.
2. Write a class `Pair<T>` holding two values of type `T`, implementing `IEnumerable<T>` so that `foreach` over a `Pair<T>` yields both values in order.
3. Explain, in terms of the `IEnumerable<T>` / `IEnumerator<T>` contract from earlier in this section, why a type can be `foreach`-able without being indexable — that is, without supporting `range[i]`.

```{rubric} Footnotes
```


[^development-reading]: Further reading: [OpenDSA CS3, Chapter Introduction: Lists](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/ListIntro.html). The worked business scenarios and guided reasoning here are original teaching examples.
